In [22]:
# ========================================+++++====
# Administração – Custo de Academia de Pilates VIVA
# ================================+++++============
# Requisitos: Python 3.9+, pandas, numpy
# Saídas: relatórios no console e CSVs opcionais em "./out/"
# --------------------------------------------

import os
from dataclasses import dataclass
from typing import Dict, List
import numpy as np
import pandas as pd

# --------------------------------------------
# CONFIGURAÇÃO — EDITE AQUI
# --------------------------------------------

# Custos fixos mensais (R$)
CUSTOS_FIXOS = {
    "Aluguel": 6500,
    "Condomínio/IPTU": 1200,
    "Energia/Água": 900,
    "Internet/Telefonia": 200,
    "Software/Plataformas": 250,
    "Marketing": 800,
    "Contabilidade": 500,
    "Limpeza/Manutenção": 600,
    "Seguros": 180,
    "Salários Administrativos": 4000,   # recepção/gestão (não instrutores)
    "Prolabore": 3000
}

# Custos variáveis por AULA
CUSTOS_VARIAVEIS_POR_AULA = {
    "Instrutor (hora/aula)": 65,      # se instrutor for variável; caso seja CLT fixo, mova para fixos
    "Materiais/Descartáveis": 5,
    "Taxas de Pagamento (%)": 0.03,   # % sobre a receita daquela aula (cartão/maquininha)
}

# Capacidade operacional
ESTACOES = 6                 # nº de alunos simultâneos por turma
AULAS_DIA = 10                # nº de turmas por dia
DIAS_MES = 26                # funcionamento no mês
OCUPACAO_MEDIA = 0.65        # 65% em média

# Planos (preço e consumo médio de aulas por aluno/mês)
PLANOS = {
    "Mensal - 1_sem":       {"preco": 260, "aulas_medias_mes": 4},
    "Mensal - 2_sem":       {"preco": 423, "aulas_medias_mes": 8},
    "Mensal - 3_sem":       {"preco": 618, "aulas_medias_mes": 12},
    "Trimestral - 1_sem":   {"preco": 245, "aulas_medias_mes": 4},
    "Trimestral - 2_sem":   {"preco": 402, "aulas_medias_mes": 8},
    "Trimestral - 3_sem":   {"preco": 588, "aulas_medias_mes": 12},
    "Semestral - 1_sem":    {"preco": 235, "aulas_medias_mes": 4},
    "Semestral - 2_sem":    {"preco": 381, "aulas_medias_mes": 8},
    "Semestral - 3_sem":    {"preco": 557, "aulas_medias_mes": 12},
    "Avulso":               {"preco": 90,  "aulas_medias_mes": 1}
}

# Mix de alunos por plano (Três alunos ativos por mês)
MIX_ALUNOS = {
    "Mensal - Contrato anual": 120,
    "Pacote 8 aulas":   90,
    "Pacote 3 aulas":   95,
    "Avulso":           95   # alunos distintos que compram avulso no mês
}

# Política de no-show (percentual de aulas pagas não realizadas)
NO_SHOW = 0.07  # 7% (receita existe, custo variável de instrutor normalmente continua)

# Exportar CSVs?
EXPORTAR_CSV = True

# --------------------------------------------
# MODELOS / FUNÇÕES
# --------------------------------------------

@dataclass
class EstudioPilates:
    custos_fixos: Dict[str, float]
    custos_variaveis_por_aula: Dict[str, float]
    estacoes: int
    aulas_dia: int
    dias_mes: int
    ocupacao_media: float
    planos: Dict[str, Dict[str, float]]
    mix_alunos: Dict[str, int]
    no_show: float

    def capacidade_teorica_aulas(self) -> int:
        # aulas realizadas no mês (turmas) independentemente de ocupação
        return self.aulas_dia * self.dias_mes

    def capacidade_teorica_vagas(self) -> int:
        # total de "assentos" (vagas) ofertados no mês
        return self.capacidade_teorica_aulas() * self.estacoes

    def vagas_ocupadas(self) -> float:
        return self.capacidade_teorica_vagas() * self.ocupacao_media

    def aulas_ocupadas(self) -> float:
        # quantidade de AULAS-ALUNO efetivamente ocupadas (cada aluno ocupa 1 estação em 1 aula)
        # é igual às vagas ocupadas
        return self.vagas_ocupadas()

    def receita_planos(self) -> pd.DataFrame:
        linhas = []
        for nome, cfg in self.planos.items():
            alunos = self.mix_alunos.get(nome, 0)
            preco = cfg["preco"]
            aulas_mes = cfg["aulas_medias_mes"]
            # Receita do plano: alunos * preço
            receita = alunos * preco
            # Aulas consumidas (estimadas) por mês
            aulas_consumidas = alunos * aulas_mes
            # Considera no-show: parte das aulas não realizadas permanece como receita, mas sem consumo de estação
            aulas_nao_realizadas = aulas_consumidas * self.no_show
            aulas_realizadas = aulas_consumidas - aulas_nao_realizadas
            linhas.append({
                "Plano": nome,
                "Alunos": alunos,
                "Preço (R$)": preco,
                "Aulas méd./aluno": aulas_mes,
                "Receita (R$)": receita,
                "Aulas consumidas": aulas_consumidas,
                "No-show (aulas)": aulas_nao_realizadas,
                "Aulas realizadas": aulas_realizadas
            })
        df = pd.DataFrame(linhas)
        return df

    def checar_capacidade(self, df_receita: pd.DataFrame) -> pd.DataFrame:
        total_aulas_realizadas = df_receita["Aulas realizadas"].sum()
        cap = self.aulas_ocupadas()
        falta_sobra = cap - total_aulas_realizadas
        return pd.DataFrame([{
            "Capacidade (vagas mês)": cap,
            "Demanda (aulas realizadas)": total_aulas_realizadas,
            "Sobra(+)/Falta(-) de vagas": falta_sobra
        }])

    def custos_fixos_total(self) -> float:
        return float(np.sum(list(self.custos_fixos.values())))

    def custo_variavel_por_aula(self, receita_media_por_aula: float) -> float:
        """
        Custo variável por AULA-ALUNO:
        - itens absolutos (R$ por aula)
        - taxas % sobre receita daquela aula
        """
        custo_abs = 0.0
        taxa_perc = 0.0
        for k, v in self.custos_variaveis_por_aula.items():
            if isinstance(v, (int, float)):
                if 0 <= v <= 0.5:   # heurística: valores <=50% tratados como porcentagem
                    taxa_perc += v
                else:
                    custo_abs += v
        return custo_abs + taxa_perc * receita_media_por_aula

    def unit_economics(self, df_receita: pd.DataFrame) -> pd.DataFrame:
        # Receita total e aulas realizadas totais
        receita_total = df_receita["Receita (R$)"].sum()
        aulas_realizadas_totais = df_receita["Aulas realizadas"].sum()

        # Receita média por aula-aluno realizada
        receita_media_por_aula = (receita_total / aulas_realizadas_totais) if aulas_realizadas_totais > 0 else 0.0

        # CV por aula
        cv_por_aula = self.custo_variavel_por_aula(receita_media_por_aula)

        # Margem de contribuição por aula e total
        margem_contrib_por_aula = receita_media_por_aula - cv_por_aula
        margem_contrib_total = margem_contrib_por_aula * aulas_realizadas_totais

        # Ponto de equilíbrio (aulas): quando margem contrib total = custos fixos
        cf_total = self.custos_fixos_total()
        aulas_break_even = cf_total / margem_contrib_por_aula if margem_contrib_por_aula > 0 else np.inf

        # Ponto de equilíbrio (alunos) aproximado: assume consumo médio ponderado de aulas/aluno
        aulas_consumidas_totais = df_receita["Aulas consumidas"].sum()
        alunos_totais = df_receita["Alunos"].sum()
        aulas_medias_por_aluno = (aulas_consumidas_totais / alunos_totais) if alunos_totais > 0 else 0.0
        alunos_break_even = (aulas_break_even / aulas_medias_por_aluno) if aulas_medias_por_aluno > 0 else np.inf

        dados = {
            "Receita total (R$)": receita_total,
            "Aulas realizadas (mês)": aulas_realizadas_totais,
            "Receita média por aula (R$)": receita_media_por_aula,
            "CV por aula (R$)": cv_por_aula,
            "Margem contrib. por aula (R$)": margem_contrib_por_aula,
            "Margem contrib. total (R$)": margem_contrib_total,
            "Custos fixos (R$)": cf_total,
            "Break-even (aulas/mês)": aulas_break_even,
            "Aulas médias por aluno": aulas_medias_por_aluno,
            "Break-even (alunos)": alunos_break_even
        }
        return pd.DataFrame([dados])

    def dres_resumo(self, df_receita: pd.DataFrame, df_unit: pd.DataFrame) -> pd.DataFrame:
        receita = df_unit.at[0, "Receita total (R$)"]
        mc_total = df_unit.at[0, "Margem contrib. total (R$)"]
        cf = df_unit.at[0, "Custos fixos (R$)"]
        ebitda = mc_total - cf
        margem_ebitda = ebitda / receita if receita > 0 else np.nan
        return pd.DataFrame([{
            "Receita (R$)": receita,
            "(-) Custos Variáveis (R$)": receita - mc_total,
            "(=) Margem de Contribuição (R$)": mc_total,
            "(-) Custos Fixos (R$)": cf,
            "(=) EBITDA (R$)": ebitda,
            "Margem EBITDA (%)": margem_ebitda * 100
        }])

    def sensibilidade(self, df_receita_base: pd.DataFrame,
                      variacao_preco: List[float]=[-0.1, 0, 0.1],
                      variacao_ocup: List[float]=[-0.1, 0, 0.1]) -> pd.DataFrame:
        """
        Avalia impacto no EBITDA com variações de preço dos planos e ocupação.
        variacao_preco/ocup: lista de variações percentuais (ex.: -0.1 = -10%)
        """
        resultados = []
        receita_base = df_receita_base["Receita (R$)"].sum()
        aulas_realizadas_base = df_receita_base["Aulas realizadas"].sum()
        for vp in variacao_preco:
            for vo in variacao_ocup:
                # Ajuste de receita por preço
                receita = receita_base * (1 + vp)
                # Ajuste de aulas realizadas por ocupação
                aulas_realizadas = aulas_realizadas_base * (1 + vo)

                # Receita média por aula ajustada
                rec_med_aula = receita / aulas_realizadas if aulas_realizadas > 0 else 0.0
                cv_por_aula = self.custo_variavel_por_aula(rec_med_aula)
                mc_por_aula = rec_med_aula - cv_por_aula
                mc_total = mc_por_aula * aulas_realizadas
                ebitda = mc_total - self.custos_fixos_total()

                resultados.append({
                    "Δ Preço": f"{vp:+.0%}",
                    "Δ Ocupação": f"{vo:+.0%}",
                    "Receita (R$)": receita,
                    "Aulas realizadas": aulas_realizadas,
                    "MC total (R$)": mc_total,
                    "EBITDA (R$)": ebitda,
                    "Margem EBITDA (%)": (ebitda/receita*100) if receita>0 else np.nan
                })
        return pd.DataFrame(resultados).sort_values(by=["Δ Preço", "Δ Ocupação"]).reset_index(drop=True)

# --------------------------------------------
# EXECUÇÃO
# --------------------------------------------

def main():
    os.makedirs("out", exist_ok=True)

    estudio = EstudioPilates(
        custos_fixos=CUSTOS_FIXOS,
        custos_variaveis_por_aula=CUSTOS_VARIAVEIS_POR_AULA,
        estacoes=ESTACOES,
        aulas_dia=AULAS_DIA,
        dias_mes=DIAS_MES,
        ocupacao_media=OCUPACAO_MEDIA,
        planos=PLANOS,
        mix_alunos=MIX_ALUNOS,
        no_show=NO_SHOW
    )

    print("\n=== CAPACIDADE ===")
    print(f"Aulas/mês (turmas): {estudio.capacidade_teorica_aulas():,.0f}")
    print(f"Vagas/mês (estações x aulas): {estudio.capacidade_teorica_vagas():,.0f}")
    print(f"Ocupação média: {OCUPACAO_MEDIA:.0%}")
    print(f"Vagas ocupadas: {estudio.vagas_ocupadas():,.0f}")

    df_receita = estudio.receita_planos()
    df_cap = estudio.checar_capacidade(df_receita)
    df_unit = estudio.unit_economics(df_receita)
    df_dre = estudio.dres_resumo(df_receita, df_unit)
    df_sens = estudio.sensibilidade(df_receita)

    print("\n=== RECEITA E CONSUMO POR PLANO ===")
    print(df_receita.to_string(index=False, formatters={
        "Preço (R$)": "R$ {:,.2f}".format,
        "Receita (R$)": "R$ {:,.2f}".format,
        "Aulas consumidas": "{:,.0f}".format,
        "No-show (aulas)": "{:,.0f}".format,
        "Aulas realizadas": "{:,.0f}".format
        
    }))

    print("\n=== CAPACIDADE x DEMANDA ===")
    print(df_cap.to_string(index=False, formatters={
        "Capacidade (vagas mês)": "{:,.0f}".format,
        "Demanda (aulas realizadas)": "{:,.0f}".format,
        "Sobra(+)/Falta(-) de vagas": "{:,.0f}".format
    }))

    print("\n=== UNIT ECONOMICS / BREAK-EVEN ===")
    print(df_unit.to_string(index=False, formatters={
        "Receita total (R$)": "R$ {:,.2f}".format,
        "Aulas realizadas (mês)": "{:,.0f}".format,
        "Receita média por aula (R$)": "R$ {:,.2f}".format,
        "CV por aula (R$)": "R$ {:,.2f}".format,
        "Margem contrib. por aula (R$)": "R$ {:,.2f}".format,
        "Margem contrib. total (R$)": "R$ {:,.2f}".format,
        "Custos fixos (R$)": "R$ {:,.2f}".format,
        "Break-even (aulas/mês)": "{:,.0f}".format,
        "Aulas médias por aluno": "{:,.2f}".format,
        "Break-even (alunos)": "{:,.0f}".format
    }))

    print("\n=== DRE GERENCIAL (MENSAL) ===")
    print(df_dre.to_string(index=False, formatters={
        "Receita (R$)": "R$ {:,.2f}".format,
        "(-) Custos Variáveis (R$)": "R$ {:,.2f}".format,
        "(=) Margem de Contribuição (R$)": "R$ {:,.2f}".format,
        "(-) Custos Fixos (R$)": "R$ {:,.2f}".format,
        "(=) EBITDA (R$)": "R$ {:,.2f}".format,
        "Margem EBITDA (%)": "{:,.1f}%".format
    }))

    print("\n=== SENSIBILIDADE (Preço x Ocupação) ===")
    print(df_sens.to_string(index=False, formatters={
        "Receita (R$)": "R$ {:,.2f}".format,
        "Aulas realizadas": "{:,.0f}".format,
        "MC total (R$)": "R$ {:,.2f}".format,
        "EBITDA (R$)": "R$ {:,.2f}".format,
        "Margem EBITDA (%)": "{:,.1f}%".format
    }))

    if EXPORTAR_CSV:
        df_receita.to_csv("out/receita_planos.csv", index=False, encoding="utf-8")
        df_cap.to_csv("out/capacidade_vs_demanda.csv", index=False, encoding="utf-8")
        df_unit.to_csv("out/unit_economics.csv", index=False, encoding="utf-8")
        df_dre.to_csv("out/dre_gerencial.csv", index=False, encoding="utf-8")
        df_sens.to_csv("out/sensibilidade_preco_ocupacao.csv", index=False, encoding="utf-8")
        print('\nArquivos salvos em "./out/":')
        for arq in ["receita_planos.csv","capacidade_vs_demanda.csv","unit_economics.csv","dre_gerencial.csv","sensibilidade_preco_ocupacao.csv"]:
            print(" -", arq)

if __name__ == "__main__":
    main()

print("-"* 115)
print("DRE = (Receita - Custos) ou (Margem de Contribuição - Custos Fixos) = EBITIDA")
print("DRE = Demonstração do Resultado do Exercício")
print("EBITIDA = Lucros Antes de Juros, Impostos, Depreciação e Amortização")
print("-"* 115)


=== CAPACIDADE ===
Aulas/mês (turmas): 260
Vagas/mês (estações x aulas): 1,560
Ocupação média: 65%
Vagas ocupadas: 1,014

=== RECEITA E CONSUMO POR PLANO ===
             Plano  Alunos Preço (R$)  Aulas méd./aluno Receita (R$) Aulas consumidas No-show (aulas) Aulas realizadas
    Mensal - 1_sem       0  R$ 260.00                 4      R$ 0.00                0               0                0
    Mensal - 2_sem       0  R$ 423.00                 8      R$ 0.00                0               0                0
    Mensal - 3_sem       0  R$ 618.00                12      R$ 0.00                0               0                0
Trimestral - 1_sem       0  R$ 245.00                 4      R$ 0.00                0               0                0
Trimestral - 2_sem       0  R$ 402.00                 8      R$ 0.00                0               0                0
Trimestral - 3_sem       0  R$ 588.00                12      R$ 0.00                0               0                0
 Semestr